# Standalone APUSH HF GPU Eval

This notebook does **not** require running from the local `slm/` repo. It is meant for Colab / notebook GPU execution.

It does four things:
1. Pulls the APUSH eval prompt and source data from GitHub.
2. Loads the Qwen base model and APUSH LoRA adapter from Hugging Face.
3. Calls the teacher and judge through the API token.
4. Runs the same base-vs-tuned-vs-teacher litmus protocol inline in the notebook.

Default models:
- base: `unsloth/Qwen3-4B`
- tuned adapter: `rohanpalviela/qwen3-4b-apush-v3-clean-lora`
- teacher: `claude-group/claude-opus-4-8`
- judge: `openai-group/gpt-5.5`

In [ ]:
# Runtime setup. Run this first.
import json, math, os, re, statistics, subprocess, sys, time, urllib.error, urllib.parse, urllib.request
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

WORKDIR = Path('/content/apush_eval') if Path('/content').exists() else Path.cwd() / 'apush_eval_workspace'
WORKDIR.mkdir(parents=True, exist_ok=True)
print('workdir:', WORKDIR)
print('python:', sys.executable)

## Configuration

Set `PROMPTLENS_API_KEY` for teacher generation and judging. Set `HF_TOKEN` only if the base or adapter repo is private/gated.

If your repo branch is not `main`, change `GITHUB_REF`.

In [ ]:
# Data/prompt source
GITHUB_OWNER_REPO = 'RohanPalivela/slm'
GITHUB_REF = 'main'
RAW_BASE = f'https://raw.githubusercontent.com/{GITHUB_OWNER_REPO}/{GITHUB_REF}'

# HF local inference
BASE_MODEL_ID = 'unsloth/Qwen3-4B'  # exact base used in the training notebook
APUSH_ADAPTER_ID = 'rohanpalviela/qwen3-4b-apush-v3-clean-lora'
LOAD_IN_4BIT = False  # faster on L4/A100; set True only if you hit OOM
MAX_NEW_TOKENS = 768    # smoke/eval speed lever; raise to 1536 if outputs truncate
TEMPERATURE = 0.0       # deterministic and usually less rambly for eval
STRIP_EMPTY_THINK_PREFILL = True  # remove Qwen3's empty no-think <think></think> prefill before generation

# API models
API_KEY_ENV = 'PROMPTLENS_API_KEY'
API_BASE_URL = 'https://tfy.promptlens.trilogy.com/v1'
TEACHER_MODEL = 'claude-group/claude-opus-4-8'
JUDGE_MODEL = 'openai-group/gpt-5.5'

# Eval scope
SPLIT = 'EVAL_HELDOUT'
ARCHETYPES = ['CAUSE_OF_SOURCE', 'EFFECT_OF_SOURCE']
DIFFICULTY = 'operational / test-day'
INCLUDE_FEWSHOT = False

# Optional: read tokens from Colab Secrets if available.
try:
    from google.colab import userdata
    for _name in ['PROMPTLENS_API_KEY', 'HF_TOKEN']:
        if not os.environ.get(_name):
            try:
                _value = userdata.get(_name)
            except Exception:
                _value = None
            if _value:
                os.environ[_name] = _value
except Exception:
    pass

# Start tiny. Full heldout eval: RUNS=3, LIMIT=0, N=2 or N=6 if time allows.
RUNS = 1
LIMIT = 2       # 0 = all sources in split
N = 2           # requested items per source; notebook calls once per archetype
RUN_TEACHER = bool(os.environ.get(API_KEY_ENV))
USE_JUDGE = bool(os.environ.get(API_KEY_ENV))

print('teacher enabled:', RUN_TEACHER)
print('judge enabled:', USE_JUDGE)
if not os.environ.get(API_KEY_ENV):
    print(f'No {API_KEY_ENV}: base+tuned can still run with programmatic checks, but teacher/judge are disabled.')

## Install GPU Inference Dependencies

On Colab, use a GPU runtime before running this. Keep `LOAD_IN_4BIT=False` on L4/A100 unless you hit OOM; fp16/bf16 is usually faster than bitsandbytes 4-bit for this 4B model.

In [ ]:
INSTALL_DEPS = True
if INSTALL_DEPS:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '-U',
           'transformers', 'accelerate', 'peft', 'bitsandbytes', 'sentencepiece', 'huggingface_hub']
    print('+', ' '.join(cmd))
    subprocess.check_call(cmd)

import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
else:
    print('WARNING: no CUDA GPU. This will be very slow.')

## Pull Prompt and Eval Data

This pulls only the files needed for eval. It does not clone or import the local repo.

In [ ]:
def fetch_text(path, *, required=True):
    url = f'{RAW_BASE}/{path}'
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            text = r.read().decode('utf-8')
    except Exception as e:
        if required:
            raise RuntimeError(f'Failed to fetch {url}. If this branch is wrong, set GITHUB_REF.') from e
        return None
    out = WORKDIR / path
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(text, encoding='utf-8')
    print('fetched', path, f'({len(text):,} chars)')
    return text

prompt_md = fetch_text('prompts/litmus_generation_prompt.md')
splits = json.loads(fetch_text('data/splits.json'))
seed_stimuli_lines = fetch_text('data/seed_stimuli.jsonl').splitlines()
seed_stimuli = [json.loads(line) for line in seed_stimuli_lines if line.strip()]
print('sources:', len(seed_stimuli))
print('splits:', ', '.join(splits['splits']))

## Inline Prompt Loader, JSON Parser, and Programmatic Checks

In [ ]:
def fenced_block_after(md_text, header_substr):
    lines = md_text.splitlines(); last_header = ''; i = 0
    while i < len(lines):
        line = lines[i]
        if line.lstrip().startswith('#'): last_header = line.lower()
        if line.lstrip().startswith('```'):
            body = []; i += 1
            while i < len(lines) and not lines[i].lstrip().startswith('```'):
                body.append(lines[i]); i += 1
            if header_substr.lower() in last_header: return '\n'.join(body)
        i += 1
    return None

@dataclass
class LitmusPrompt:
    system: str
    user: str
    fewshot: str | None = None
    @classmethod
    def from_markdown(cls, md_text):
        system = fenced_block_after(md_text, '## system')
        user = fenced_block_after(md_text, '## user')
        fewshot = fenced_block_after(md_text, 'few-shot')
        if not system or not user: raise ValueError('could not parse SYSTEM/USER fenced blocks')
        return cls(system.strip(), user.strip(), fewshot.strip() if fewshot else None)
    def build(self, *, source, attribution, note, n, archetypes, difficulty, include_fewshot=False):
        system = self.system
        if include_fewshot and self.fewshot: system += '\n\nFEW-SHOT EXEMPLARS:\n' + self.fewshot
        user = (self.user.replace('{{SOURCE}}', source).replace('{{ATTRIBUTION}}', attribution)
                .replace('{{NOTE}}', note or '(none)').replace('{{N}}', str(n))
                .replace('{{ARCHETYPES}}', archetypes).replace('{{DIFFICULTY}}', difficulty))
        return system, user

ITEM_LIST_KEYS = ('items', 'questions', 'mcqs', 'data', 'results', 'output')
ARCHETYPE_ALIASES = {'cause_of':'CAUSE_OF_SOURCE','effect_immediate':'EFFECT_OF_SOURCE','effect_longterm':'LONGTERM_LEGACY','continuation_change':'CONTINUITY_OR_CHANGE','reflects_illustrates':'DEVELOPMENT_ILLUSTRATED','context_response_to':'CONTEXT_SITUATION','influenced_by':'CONTEXT_INFLUENCED_BY','purpose_intended_to':'SOURCE_POV_PURPOSE','point_of_view':'SOURCE_POV_PURPOSE','evidence_supports':'EVIDENCE_SUPPORTS_CLAIM','evidence_undermines':'EVIDENCE_UNDERMINES_CLAIM','similar_effect':'COMPARATIVE_ANALOG','differs_from':'COMPETING_INTERPRETATIONS'}

def normalize_items(obj):
    if isinstance(obj, dict):
        for key in ITEM_LIST_KEYS:
            if isinstance(obj.get(key), list): return normalize_items(obj[key])
        return [obj]
    if isinstance(obj, list): return [x for x in obj if isinstance(x, dict)]
    return []

def extract_items(text):
    if not text: return []
    t = re.sub(r'\s*```$', '', re.sub(r'^```[a-zA-Z]*\s*', '', text.strip()))
    try: return normalize_items(json.loads(t))
    except json.JSONDecodeError: pass
    for open_ch, close_ch in (('[', ']'), ('{', '}')):
        start = t.find(open_ch)
        if start == -1: continue
        depth = 0; in_str = False; esc = False
        for j, c in enumerate(t[start:], start):
            if in_str:
                if esc: esc = False
                elif c == '\\': esc = True
                elif c == '"': in_str = False
                continue
            if c == '"': in_str = True
            elif c == open_ch: depth += 1
            elif c == close_ch:
                depth -= 1
                if depth == 0:
                    try: return normalize_items(json.loads(t[start:j+1]))
                    except json.JSONDecodeError: break
    return []

def canonicalize_item_archetype(item, requested_archetype=None):
    out = dict(item); raw = out.get('archetype')
    if raw is not None and '_model_archetype' not in out: out['_model_archetype'] = raw
    if requested_archetype:
        out['archetype'] = requested_archetype; out['_requested_archetype'] = requested_archetype
    elif isinstance(raw, str): out['archetype'] = ARCHETYPE_ALIASES.get(raw.strip(), raw)
    return out

ABSOLUTE_OR_ALLNONE = re.compile(r'\b(all|none)\s+of\s+the\s+above\b|\balways\b|\bnever\b', re.I)
YEAR_RE = re.compile(r'\b(1[4-9]\d\d|20\d\d)s?\b')
PARENTHETICAL_DATE_LABEL = re.compile(r'\(\s*(?:c\.\s*)?(?:1[4-9]\d\d|20\d\d)(?:\s*[-–]\s*(?:c\.\s*)?(?:\d{2,4}|present))?\s*\)', re.I)
VALID_TRAP_TYPES = {'WRONG_ERA','TRUE_BUT_IRRELEVANT','SCOPE_MISMATCH','PARTIALLY_TRUE'}
CAUSE_ARCHETYPES = {'CAUSE_OF_SOURCE','CONTEXT_SITUATION','CONTEXT_INFLUENCED_BY'}
EFFECT_ARCHETYPES = {'EFFECT_OF_SOURCE','LONGTERM_LEGACY','COMPARATIVE_ANALOG','CONTINUITY_OR_CHANGE'}

def strip_label(opt): return re.sub(r'^\s*[A-D][).:]?\s+', '', opt or '').strip()
def answer_index(item):
    a = str(item.get('answer', '')).strip().upper()
    return 'ABCD'.index(a[0]) if a and a[0] in 'ABCD' else None
def norm_text(s): return re.sub(r'\s+', ' ', (s or '').lower()).strip()
def years(text): return [int(y) for y in YEAR_RE.findall(text or '')]
def source_year(source):
    y = source.get('year')
    if isinstance(y, list) and y: return int(y[0])
    if isinstance(y, int): return y
    found = years(source.get('attribution', '')); return found[0] if found else None

def date_direction(item, source):
    src_year = source_year(source); arch = item.get('archetype', '')
    if src_year is None or (arch not in CAUSE_ARCHETYPES and arch not in EFFECT_ARCHETYPES): return 'unknown'
    ys = [y for y in years(item.get('answer_dating', '')) if y != src_year]
    if not ys: return 'unknown'
    return ('fail' if min(ys) > src_year else 'pass') if arch in CAUSE_ARCHETYPES else ('fail' if max(ys) < src_year else 'pass')

def source_leak(item, source):
    idx = answer_index(item); opts = item.get('options', [])
    if idx is None or idx >= len(opts): return False
    ans = norm_text(strip_label(opts[idx])); return len(ans) >= 40 and ans in norm_text(source.get('text', ''))

def rationale_complete(item):
    rat = item.get('rationale'); return isinstance(rat, dict) and all(k in rat and str(rat.get(k, '')).strip() for k in 'ABCD')
def rationale_marks_key(item):
    idx = answer_index(item); rat = item.get('rationale')
    if idx is None or not isinstance(rat, dict): return False
    txt = str(rat.get('ABCD'[idx], '')).strip().lower()
    if txt in {'correct','correct.'} or txt.startswith('correct:'): return True
    return bool(txt) and not any(txt.startswith(t.lower()) for t in VALID_TRAP_TYPES)

def run_checks(item, source):
    opts = item.get('options', []); idx = answer_index(item); labels = [strip_label(o) for o in opts] if opts else []
    four_options = isinstance(opts, list) and len(opts) == 4
    one_key = idx is not None and 0 <= idx < len(opts)
    no_all_none = not any(ABSOLUTE_OR_ALLNONE.search(o or '') for o in opts)
    homogeneous = True
    if four_options and one_key and labels:
        lens = sorted(len(l) for k, l in enumerate(labels) if k != idx); homogeneous = len(labels[idx]) <= 1.7 * (lens[len(lens)//2] or 1)
    traps = [str(t).strip().upper() for t in (item.get('trap_types', []) or []) if str(t).strip()]
    trap_count_3 = len(traps) == 3; trap_types_valid = bool(traps) and all(t in VALID_TRAP_TYPES for t in traps)
    trap_diversity = len(set(t for t in traps if t != 'CORRECT')) >= 2; wrong_era_le1 = sum(1 for t in traps if t == 'WRONG_ERA') <= 1
    no_option_dates = not any(PARENTHETICAL_DATE_LABEL.search(o or '') for o in opts)
    leak = source_leak(item, source); date = date_direction(item, source); schema_ok = rationale_complete(item) and rationale_marks_key(item)
    disqualifying_ok = four_options and one_key and no_all_none and not leak and date != 'fail'
    craft_ok = trap_count_3 and trap_types_valid and trap_diversity and wrong_era_le1 and no_option_dates and schema_ok
    return {'four_options': four_options, 'one_key': one_key, 'no_all_none_absolute': no_all_none, 'homogeneous_length': homogeneous,
            'trap_count_3': trap_count_3, 'trap_types_valid': trap_types_valid, 'trap_diversity_ge2': trap_diversity,
            'wrong_era_le1': wrong_era_le1, 'no_parenthetical_option_dates': no_option_dates, 'rationale_complete': rationale_complete(item),
            'rationale_marks_key': rationale_marks_key(item), 'source_leak': leak, 'date_direction': date,
            'disqualifying_ok': disqualifying_ok, 'schema_ok': schema_ok, 'craft_ok': craft_ok}

prompt = LitmusPrompt.from_markdown(prompt_md)
by_id = {s['id']: s for s in seed_stimuli}
source_ids = splits['splits'][SPLIT]['source_ids']
sources = [by_id[sid] for sid in source_ids if sid in by_id]
if LIMIT: sources = sources[:LIMIT]
print('loaded split:', SPLIT, 'sources:', len(sources), [s['id'] for s in sources[:5]])

## Providers: HF Local GPU + API Teacher/Judge

The HF engine loads the base model once, attaches the APUSH adapter once, and uses `disable_adapter()` for base generations. That avoids reloading a 4B model for every model comparison.

In [ ]:
class ProviderError(Exception): pass

def post_json(url, headers, payload, timeout=240):
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(url, data=data, headers=headers, method='POST')
    last = None
    for attempt in range(3):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r: return json.load(r)
        except urllib.error.HTTPError as e:
            body = e.read().decode('utf-8', 'replace'); last = ProviderError(f'HTTP {e.code}: {body[:800]}')
            if e.code in (429,500,502,503,529): time.sleep(2*(attempt+1)); continue
            raise last
        except Exception as e:
            last = ProviderError(f'network error: {e}'); time.sleep(2*(attempt+1))
    raise last or ProviderError('unknown API error')

def api_generate(model, system, user, *, max_tokens=4096, temperature=0.7, include_temperature=True, token_param='max_tokens'):
    key = os.environ.get(API_KEY_ENV, '')
    if not key: raise ProviderError(f'missing {API_KEY_ENV}')
    payload = {'model': model, 'messages': [{'role':'system','content':system}, {'role':'user','content':user}], token_param: max_tokens}
    if include_temperature: payload['temperature'] = temperature
    headers = {'Content-Type':'application/json', 'Authorization':f'Bearer {key}'}
    try: resp = post_json(API_BASE_URL.rstrip() + '/chat/completions', headers, payload)
    except ProviderError as e:
        s = str(e).lower()
        if 'http 400' in s and 'temperature' in s and include_temperature:
            return api_generate(model, system, user, max_tokens=max_tokens, temperature=temperature, include_temperature=False, token_param=token_param)
        if 'http 400' in s and 'max_tokens' in s and token_param == 'max_tokens':
            return api_generate(model, system, user, max_tokens=max_tokens, temperature=temperature, include_temperature=include_temperature, token_param='max_completion_tokens')
        raise
    return ((resp.get('choices') or [{}])[0].get('message') or {}).get('content') or ''

EMPTY_THINK_RE = re.compile(r'<think>\s*</think>\s*', re.I | re.S)
def strip_empty_think_block(text):
    return EMPTY_THINK_RE.sub('', text)

class HFLocalEngine:
    def __init__(self): self.loaded = False; self.tokenizer = None; self.model = None
    def load(self):
        if self.loaded: return
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import PeftModel
        token = os.environ.get('HF_TOKEN') or None; common = {'token': token, 'trust_remote_code': True}
        try: self.tokenizer = AutoTokenizer.from_pretrained(APUSH_ADAPTER_ID, **common)
        except Exception: self.tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, **common)
        if self.tokenizer.pad_token_id is None and self.tokenizer.eos_token_id is not None: self.tokenizer.pad_token = self.tokenizer.eos_token
        torch.backends.cuda.matmul.allow_tf32 = True
        try: torch.set_float32_matmul_precision('high')
        except Exception: pass
        dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        kwargs = {'device_map':'auto', 'torch_dtype':dtype, **common}
        if LOAD_IN_4BIT:
            from transformers import BitsAndBytesConfig
            kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
        print('loading base:', BASE_MODEL_ID); base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **kwargs)
        print('attaching adapter:', APUSH_ADAPTER_ID); self.model = PeftModel.from_pretrained(base, APUSH_ADAPTER_ID, token=token)
        self.model.eval(); self.loaded = True; print('loaded model')
    def render_prompt(self, system, user, *, think=False):
        messages = [{'role':'system','content':system}, {'role':'user','content':user}]
        try:
            rendered = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=think)
        except TypeError:
            rendered = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        if not think and STRIP_EMPTY_THINK_PREFILL:
            rendered = strip_empty_think_block(rendered)
        return rendered
    def generate(self, system, user, *, tuned, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS):
        self.load(); prompt_text = self.render_prompt(system, user, think=False); device = next(self.model.parameters()).device
        inputs = self.tokenizer(prompt_text, return_tensors='pt').to(device)
        do_sample = temperature is not None and float(temperature) > 0
        gen_kwargs = {'max_new_tokens': max_new_tokens, 'do_sample': do_sample, 'use_cache': True, 'pad_token_id': self.tokenizer.pad_token_id or self.tokenizer.eos_token_id, 'eos_token_id': self.tokenizer.eos_token_id}
        if do_sample: gen_kwargs.update({'temperature': float(temperature), 'top_p': 0.95})
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.time()
        with torch.inference_mode():
            if tuned: out = self.model.generate(**inputs, **gen_kwargs)
            else:
                with self.model.disable_adapter(): out = self.model.generate(**inputs, **gen_kwargs)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        dt = max(time.time() - t0, 1e-6)
        new_tokens = out[0, inputs['input_ids'].shape[-1]:]
        print(f"gen_stats tuned={tuned} prompt_tok={inputs['input_ids'].shape[-1]} new_tok={new_tokens.shape[-1]} tok_s={new_tokens.shape[-1]/dt:.1f}")
        text = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        return re.sub(r'^\s*<think>\s*</think>\s*', '', text).strip()

hf_engine = HFLocalEngine()

## Check HF Access and Chat Template

In [ ]:
from huggingface_hub import model_info
for repo_id in [BASE_MODEL_ID, APUSH_ADAPTER_ID]:
    info = model_info(repo_id, token=os.environ.get('HF_TOKEN') or None)
    names = [s.rfilename for s in info.siblings]
    interesting = [n for n in names if n.endswith(('adapter_config.json','adapter_model.safetensors','config.json','tokenizer_config.json'))]
    print(repo_id, interesting[:10])
from transformers import AutoTokenizer
try: tok = AutoTokenizer.from_pretrained(APUSH_ADAPTER_ID, token=os.environ.get('HF_TOKEN') or None, trust_remote_code=True)
except Exception: tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ.get('HF_TOKEN') or None, trust_remote_code=True)
msgs = [{'role':'system','content':'Return only JSON.'}, {'role':'user','content':'Say OK as a JSON array.'}]
try: rendered = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
except TypeError: rendered = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
empty_think = bool(re.search(r'<think>\s*</think>', rendered, re.I | re.S))
nonempty_think = any(span.strip() for span in re.findall(r'<think>(.*?)</think>', rendered, flags=re.I | re.S))
rendered_for_generation = strip_empty_think_block(rendered) if STRIP_EMPTY_THINK_PREFILL else rendered
print('--- tokenizer-rendered prompt ---')
print(rendered[:1200])
print('--- prompt actually used for generation ---')
print(rendered_for_generation[:1200])
print('empty no-think prefill:', empty_think)
print('nonempty thinking text:', nonempty_think)
print('strip empty think prefill:', STRIP_EMPTY_THINK_PREFILL)
print('contains tool scaffold:', '<tools>' in rendered or 'tool_call' in rendered)


## Judge Rubric

In [ ]:
JUDGE_SYSTEM = """You are a senior AP U.S. History assessment expert grading a single machine-generated stimulus-based multiple-choice question. You know U.S. history well and you are NOT the model that wrote the item.

Grade like a real test-development reviewer: be STRICT about correctness (is the keyed answer right, is it uniquely best, does the skill match the command phrase) but FAIR about craft. An item a real APUSH exam would actually use is expert-grade even if a distractor could be marginally stronger. Do NOT invent flaws or hold items to a standard of perfection the operational test itself does not meet.

You will receive the SOURCE (stimulus) and the ITEM (stem, 4 options, keyed answer, rationale). Judge ONLY against the source + standard APUSH knowledge.

CALIBRATION:
- distractors_period_plausible is TRUE unless a distractor is genuinely absurd, fabricated, or off-topic filler.
- spec_adherence: 2 = operationally usable; 1 = genuine craft weakness short of disqualifying; 0 = disqualifying flaw.

Return ONLY a JSON object with these fields:
{"requires_outside_knowledge": true|false, "every_distractor_named_trap": true|false, "distractors_period_plausible": true|false, "skill_matches_command_phrase": true|false, "key_historically_correct": true|false, "key_uniquely_best": true|false, "single_best_answer": true|false, "spec_adherence": 0|1|2, "distractor_craft": 0|1|2, "outside_knowledge_skill_fit": 0|1|2, "notes": "<one sentence>"}"""

def fmt_options(opts): return '\n'.join(f"  {'ABCD'[i] if i < 4 else '?'}) {o}" for i, o in enumerate(opts or []))
def build_judge_prompt(source, item):
    user = f"""SOURCE ({source.get('attribution', '')}):\n\"\"\"\n{source.get('text', '')}\n\"\"\"\n\nITEM:\narchetype: {item.get('archetype', '')}\nstem: {item.get('stem', '')}\noptions:\n{fmt_options(item.get('options', []))}\nkeyed answer: {item.get('answer', '')}\nanswer_dating: {item.get('answer_dating', '')}\nrationale: {json.dumps(item.get('rationale', {}), ensure_ascii=False)}\n\nGrade it. Return ONLY the JSON object."""
    return JUDGE_SYSTEM, user

def normalize_judgment(j):
    def b(k): return bool(j.get(k, False))
    def g(k):
        try: return int(j.get(k, 0))
        except Exception: return 0
    out = {'requires_outside_knowledge':b('requires_outside_knowledge'),'every_distractor_named_trap':b('every_distractor_named_trap'),'distractors_period_plausible':b('distractors_period_plausible'),'skill_matches_command_phrase':b('skill_matches_command_phrase'),'key_historically_correct':b('key_historically_correct'),'key_uniquely_best':b('key_uniquely_best'),'single_best_answer':b('single_best_answer'),'spec_adherence':g('spec_adherence'),'distractor_craft':g('distractor_craft'),'outside_knowledge_skill_fit':g('outside_knowledge_skill_fit'),'notes':str(j.get('notes',''))[:300]}
    out['key_valid'] = out['key_historically_correct'] and out['key_uniquely_best']; return out

def judge_item(source, item):
    if not USE_JUDGE: return None
    system, user = build_judge_prompt(source, item)
    raw = api_generate(JUDGE_MODEL, system, user, max_tokens=2048, temperature=0.0, include_temperature=False, token_param='max_completion_tokens')
    parsed = extract_items(raw)
    return normalize_judgment(parsed[0] if parsed else {'notes':'judge returned unparseable output'})

def near_grade(prog_ok, j):
    if j is None: return None
    judge_disq = (j['requires_outside_knowledge'] and j['every_distractor_named_trap'] and j['distractors_period_plausible'] and j['skill_matches_command_phrase'] and j['single_best_answer'] and j['key_valid'])
    dims_ok = min(j['spec_adherence'], j['distractor_craft'], j['outside_knowledge_skill_fit']) >= 1
    return bool(prog_ok and judge_disq and dims_ok)
def expert_grade(prog_ok, j):
    ng = near_grade(prog_ok, j); return None if ng is None else bool(ng and j['spec_adherence'] == 2)

## One-Prompt Smoke Test

This loads the HF model and APUSH adapter. The first run is slow because it downloads weights and warms the GPU.

In [ ]:
source = sources[0]
arch = ARCHETYPES[0]
system, user = prompt.build(source=source['text'], attribution=source.get('attribution', ''), note='', n=1, archetypes=arch, difficulty=DIFFICULTY, include_fewshot=INCLUDE_FEWSHOT)
raw = hf_engine.generate(system, user, tuned=True, temperature=0.0, max_new_tokens=MAX_NEW_TOKENS)
print('--- raw tuned output ---')
print(raw[:4000])
print('\\nformat flags:', {'starts_json_array': raw.strip().startswith('['), 'contains_think': '<think' in raw.lower() or '</think' in raw.lower(), 'contains_markdown_fence': '```' in raw})
items = [canonicalize_item_archetype(it, requested_archetype=arch) for it in extract_items(raw)]
print('parsed items:', len(items))
if items:
    print(json.dumps(items[0], indent=2)[:3000])
    print(json.dumps(run_checks(items[0], source), indent=2))

## Inline Eval Runner

Runs one call per `(run, source, archetype)` so each archetype is measured even if a small model emits only one complete item under JSON pressure.

In [ ]:
MODEL_ROSTER = [{'name':'qwen3-4b-base','role':'candidate','kind':'hf_base'}, {'name':'qwen3-apush-tuned','role':'candidate','kind':'hf_tuned'}]
REQUIRED_COMPARISON_MODELS = ['qwen3-4b-base', 'qwen3-apush-tuned']
if RUN_TEACHER: MODEL_ROSTER.append({'name':'teacher','role':'teacher','kind':'api_teacher'})
print(MODEL_ROSTER)

def generate_model(model, system, user):
    if model['kind'] == 'hf_base': return hf_engine.generate(system, user, tuned=False, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
    if model['kind'] == 'hf_tuned': return hf_engine.generate(system, user, tuned=True, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
    if model['kind'] == 'api_teacher': return api_generate(TEACHER_MODEL, system, user, max_tokens=8192, temperature=TEMPERATURE)
    raise ValueError(model)

def score_record(rec):
    prog = run_checks(rec['item'], rec['source'])
    j = judge_item(rec['source'], rec['item']) if USE_JUDGE else None
    craft_ok = prog['disqualifying_ok'] and prog.get('craft_ok', prog.get('wrong_era_le1', True))
    key_valid = (j['key_valid'] and prog['disqualifying_ok']) if j else None
    archetype_label_match = rec['item'].get('_model_archetype', rec['item'].get('archetype')) == rec.get('archetype')
    distillable_item_valid = (key_valid and prog.get('schema_ok') and prog.get('craft_ok') and archetype_label_match) if j else None
    return {**rec, 'prog': prog, 'judge': j, 'near_miss': near_grade(craft_ok, j),
            'expert_grade': expert_grade(craft_ok, j), 'key_valid': key_valid,
            'distillable_item_valid': distillable_item_valid,
            'archetype_label_match': archetype_label_match}

def rate(records, key):
    vals = [r[key] for r in records if r[key] is not None]
    return None if not vals else sum(1 for v in vals if v) / len(vals)
def aggregate(records):
    n = len(records); by_arch = defaultdict(list)
    for r in records: by_arch[r['item'].get('archetype', '?')].append(r)
    return {'n_items':n, 'expert_grade':rate(records,'expert_grade'), 'near_miss':rate(records,'near_miss'),
            'key_valid':rate(records,'key_valid'), 'distillable_item_valid':rate(records,'distillable_item_valid'),
            'craft_fail':sum(1 for r in records if not r['prog'].get('craft_ok', True))/n if n else 0,
            'schema_fail':sum(1 for r in records if not r['prog'].get('schema_ok', True))/n if n else 0,
            'date_fail':sum(1 for r in records if r['prog'].get('date_direction') == 'fail')/n if n else 0,
            'leak':sum(1 for r in records if r['prog'].get('source_leak'))/n if n else 0,
            'archetype_label_fail':sum(1 for r in records if not r.get('archetype_label_match', True))/n if n else 0,
            'by_arch':{a: rate(v,'near_miss') for a,v in by_arch.items()}}
def pct(x): return 'n/a' if x is None else f'{x:.0%}'

def run_eval():
    records = []; attempts = []; n_per = max(1, round(N / len(ARCHETYPES))); total = len(MODEL_ROSTER) * RUNS * len(sources) * len(ARCHETYPES); done = 0
    for model in MODEL_ROSTER:
        print('\\n===', model['name'], '===')
        for run_idx in range(RUNS):
            for source in sources:
                for arch in ARCHETYPES:
                    system, user = prompt.build(source=source['text'], attribution=source.get('attribution', ''), note='', n=n_per, archetypes=arch, difficulty=DIFFICULTY, include_fewshot=INCLUDE_FEWSHOT)
                    try:
                        raw = generate_model(model, system, user); items = extract_items(raw); err = None
                    except Exception as e:
                        raw, items, err = '', [], repr(e)
                    for it in items:
                        it = canonicalize_item_archetype(it, requested_archetype=arch)
                        records.append({'model':model['name'],'role':model['role'],'run':run_idx,'source_id':source['id'],'source':source,'archetype':arch,'item':it,'raw':raw[:5000],'error':err})
                    attempts.append({'model': model['name'], 'role': model['role'], 'kind': model['kind'], 'run': run_idx, 'source_id': source['id'], 'archetype': arch, 'n_items': len(items), 'error': err, 'raw_preview': raw[:1000], 'raw': raw})
                    done += 1; print(f"{done}/{total} {model['name']} run={run_idx} source={source['id']} arch={arch} items={len(items)}" + (f' ERR={err}' if err else ''))
    print('\\nScoring...')
    scored = []
    for i, rec in enumerate(records, 1):
        scored.append(score_record(rec))
        if i % 5 == 0 or i == len(records): print(f'scored {i}/{len(records)}')
    return scored, attempts

scored, generation_attempts = run_eval()
print('records:', len(scored))

## Results

In [ ]:
roster_names = [m['name'] for m in MODEL_ROSTER]
attempt_names = sorted({a['model'] for a in generation_attempts})
scored_names = sorted({r['model'] for r in scored})
model_names = []
for name in roster_names + attempt_names + scored_names:
    if name not in model_names:
        model_names.append(name)

missing_attempts = [m for m in REQUIRED_COMPARISON_MODELS if m not in attempt_names]
if missing_attempts:
    raise RuntimeError(f"Base-vs-tuned eval is invalid: missing generation attempts for {missing_attempts}. Rerun the Inline Eval Runner cell from a fresh runtime.")

results = {}
for model in model_names:
    model_scored = [r for r in scored if r['model'] == model]
    results[model] = aggregate(model_scored)
    calls = [a for a in generation_attempts if a['model'] == model]
    parsed = results[model]['n_items']
    zero_calls = sum(1 for a in calls if a['n_items'] == 0)
    results[model]['generation'] = {
        'calls': len(calls),
        'zero_item_calls': zero_calls,
        'parse_empty_rate': zero_calls / len(calls) if calls else None,
        'parsed_items_per_call': parsed / len(calls) if calls else None,
    }

def fmt_num(x): return 'n/a' if x is None else f'{x:.2f}'
print('| Model | Calls | Zero calls | Parsed | Items/call | Expert | Near | Key valid | Distillable | Craft fail | Schema fail | Arch label fail | Date fail | Leak |')
print('| :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |')
for model in model_names:
    agg = results[model]
    gen = agg['generation']
    print(f"| {model} | {gen['calls']} | {gen['zero_item_calls']} | {agg['n_items']} | {fmt_num(gen['parsed_items_per_call'])} | {pct(agg['expert_grade'])} | {pct(agg['near_miss'])} | {pct(agg['key_valid'])} | {pct(agg['distillable_item_valid'])} | {pct(agg['craft_fail'])} | {pct(agg['schema_fail'])} | {pct(agg['archetype_label_fail'])} | {pct(agg['date_fail'])} | {pct(agg['leak'])} |")

comparison = {'base_model': 'qwen3-4b-base', 'tuned_model': 'qwen3-apush-tuned'}
base = results.get(comparison['base_model'], {})
tuned = results.get(comparison['tuned_model'], {})
for metric in ['near_miss', 'key_valid', 'distillable_item_valid']:
    b = base.get(metric); t = tuned.get(metric)
    comparison[f'{metric}_delta_tuned_minus_base'] = None if b is None or t is None else t - b
results['_comparison'] = comparison
print('\nBase-vs-tuned comparison:', json.dumps(comparison, indent=2))

print('\nZero-item generation calls:')
zero_attempts = [a for a in generation_attempts if a['n_items'] == 0]
for a in zero_attempts[:20]:
    print(f"- {a.get('model')} {a.get('source_id', '?')} {a.get('archetype', '?')} error={a.get('error')} raw={str(a.get('raw_preview', ''))[:180]!r}")
if len(zero_attempts) > 20:
    print(f"... {len(zero_attempts) - 20} more")

out_dir = WORKDIR / 'results'; out_dir.mkdir(exist_ok=True)
(out_dir / 'summary.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
(out_dir / 'generation_attempts.json').write_text(json.dumps(generation_attempts, indent=2, ensure_ascii=False), encoding='utf-8')
with (out_dir / 'items.jsonl').open('w', encoding='utf-8') as f:
    for r in scored:
        slim = {k:v for k,v in r.items() if k not in ('source','item')}; slim['item'] = dict(r['item'])
        f.write(json.dumps(slim, ensure_ascii=False) + '\n')
print('wrote:', out_dir)


## Full Run Settings

After the smoke run is clean, change:

```python
RUNS = 3
LIMIT = 0
N = 6
MAX_NEW_TOKENS = 768  # start here; raise only if outputs are cut off
```

Then rerun from **Inline Eval Runner** onward. The first HF generation downloads/loads weights; subsequent cells reuse the loaded model in memory. On L4, keep `LOAD_IN_4BIT=False` unless you OOM; fp16/bf16 is usually faster than bitsandbytes 4-bit for a 4B model.